# 08 – 100 m: Modelldaten und Annahmen prüfen

Feste Auflösung: **100 m**. Eingabeprofil: **100m_legacy**.
Der historische Lauf wird aus der flachen Datenablage gelesen; `grid_id_100m` wird beim Lesen als `grid_id` verwendet.
Die Quelldateien werden nicht umgeschrieben; es erfolgt kein Neuaufbau von Routing oder Datengrundlage.
Poisson/NB2 werden auf den vorhandenen Daten neu geschätzt. Die 26 NB2-Prüfungen und die bisherigen Survival-Eingangsprüfungen bleiben erhalten.

Ergebnisse stehen direkt unter jeder Prüfzelle; Herkunft, Ausschlüsse und Zusammenfassung stehen ganz unten.
**Einmal ausführen und mit Outputs speichern (Strg+S)**; anschließend zum Nachlesen ohne Neuberechnung öffnen.
Fehlende Survival-Produkte werden separat ausgewiesen und blockieren die NB2-Checkliste nicht.
OK = geprüft/unauffällig, HINWEIS = Befund/Entscheidung, PRÜFEN = Auffälligkeit, OFFEN = nicht auswertbar.

## Paths and analysis period


In [ ]:
from pathlib import Path
import json
import platform
from datetime import datetime, timezone

import duckdb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy import stats


PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == "ANAL":
    PROJECT_DIR = PROJECT_DIR.parent
if not (PROJECT_DIR / "ANAL").is_dir():
    raise RuntimeError("Start the notebook from the repository root or the ANAL directory.")

# Fester Lauf: keine automatische Suche in anderen Datenablagen.
CELL_SIZE = 100
RESOLUTION = f"{CELL_SIZE}m"
LEGACY_INPUTS = True
INPUT_PROFILE = "100m_legacy"
DATA_DIR = PROJECT_DIR / "ANAL" / "data"
FEATURE_DIR = DATA_DIR / "routing" / "features"
PANEL_PATH = DATA_DIR / f"raster_quarter_panel_{RESOLUTION}.parquet"
FIRMS_PATH = DATA_DIR / f"firms_assigned_{RESOLUTION}.geoparquet"
BIRTHS_BY_GROUP_PATH = DATA_DIR / f"births_by_fachgruppe_{RESOLUTION}.parquet"
RASTER_PATH = DATA_DIR / f"raster_{RESOLUTION}_styria.geoparquet"
RESULT_DIR = PROJECT_DIR / "ANAL" / "data" / "assumptions" / INPUT_PROFILE
CACHE_DIR = RESULT_DIR / "cache"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR = 2016
END_YEAR = 2025
LAG_YEAR = 2015

print(f"Resolution: {RESOLUTION}; data folder: {DATA_DIR}")
print(f"Project: {PROJECT_DIR}")
print(f"Analysis period: {START_YEAR}–{END_YEAR}")
print(f"Model results: {RESULT_DIR}")


# Eine einfache Ergebnisliste für die Zusammenfassung am Notebook-Ende.
checklist = []
def report(check, status, result):
    checklist[:] = [row for row in checklist if row["Prüfpunkt"] != check]
    checklist.append({"Prüfpunkt": check, "Status": status, "Ergebnis": result})
    print(f"{check} | {status}: {result}")

# Nur zusätzliche Sensitivitätsmodelle nutzen eine feste Rasterzellen-Stichprobe.
# Poisson und NB2 der Hauptdiagnostik verwenden alle gültigen Modellzeilen.
ROBUSTNESS_CELLS = 2000
MORAN_PERMUTATIONS = 999  # Enough resolution for Holm correction across 40 quarters.
RANDOM_SEED = 42

In [ ]:
compatibility_mappings = []

def parquet_columns(path):
    if not path.exists():
        raise FileNotFoundError(path)
    return set(pq.ParquetFile(path).schema_arrow.names)

def grid_column(path):
    columns = parquet_columns(path)
    if "grid_id" in columns:
        return "grid_id"
    if LEGACY_INPUTS and "grid_id_100m" in columns:
        mapping = {"Datei": str(path.relative_to(PROJECT_DIR)),
                   "Quelle": "grid_id_100m", "Verwendet": "grid_id"}
        if mapping not in compatibility_mappings:
            compatibility_mappings.append(mapping)
            print(f"Kompatibilität: {mapping['Datei']}: grid_id_100m -> grid_id (nur im Speicher)")
        return "grid_id_100m"
    raise ValueError(f"{path} is missing columns: ['grid_id']")

def require_columns(path, required):
    available = parquet_columns(path)
    if "grid_id" in required:
        grid_column(path)
        available = available | {"grid_id"}
    missing = sorted(set(required) - available)
    if missing:
        raise ValueError(f"{path} is missing columns: {missing}")

def read_model_parquet(path, columns):
    actual_grid = grid_column(path) if "grid_id" in columns else "grid_id"
    require_columns(path, columns)
    requested = [actual_grid if c == "grid_id" else c for c in columns]
    return pd.read_parquet(path, columns=requested).rename(columns={actual_grid: "grid_id"})

input_files = pd.DataFrame([
    {"input": "Gründungspanel (Pflicht)", "path": PANEL_PATH, "exists": PANEL_PATH.exists()},
    {"input": "Firmen (Gebiets-/Überlebensprüfungen)", "path": FIRMS_PATH, "exists": FIRMS_PATH.exists()},
    {"input": "Raster (Gebiets-/Raumprüfungen)", "path": RASTER_PATH, "exists": RASTER_PATH.exists()},
])
display(input_files)
require_columns(PANEL_PATH, ["grid_id", "municipality_id", "year", "quarter", "period", "births", "active_firms_tminus1"])
print("Quellprodukte werden nur gelesen. Fehlende Kernvariablen werden nicht imputiert.")

## Checkliste 1: Daten und abhängige Variable (vor der Bereinigung)

### 1.1 Zählvariable (Pflicht)

In [ ]:
raw_panel = pd.read_parquet(PANEL_PATH, columns=[
    "grid_id", "municipality_id", "year", "quarter", "period", "births",
    "active_firms_tminus1",
])
raw_panel = raw_panel.loc[raw_panel.year.between(START_YEAR, END_YEAR)].copy()
raw_y = pd.to_numeric(raw_panel.births, errors="coerce")
invalid_y = ~np.isfinite(raw_y) | (raw_y < 0) | (raw_y != np.floor(raw_y))
report("1.1", "PRÜFEN" if invalid_y.any() else "OK",
       f"Ungültige/fehlende Zählwerte: {invalid_y.sum():,} von {len(raw_y):,}.")
if invalid_y.any():
    display(raw_panel.loc[invalid_y].head(10))
    raise ValueError("Ungültiges y zuerst in den Quelldaten korrigieren.")

### 1.2 Panelstruktur und fehlende Quartale (Pflicht)

In [ ]:
expected_periods = pd.period_range(f"{START_YEAR}Q1", f"{END_YEAR}Q4", freq="Q").astype(str)
duplicate_rows = raw_panel.duplicated(["grid_id", "year", "quarter"]).sum()
invalid_keys = raw_panel[["grid_id", "year", "quarter", "period"]].isna().any(axis=1)
invalid_keys |= ~raw_panel.quarter.isin([1, 2, 3, 4])
invalid_keys |= ~raw_panel.period.eq(raw_panel.year.astype(str) + "Q" + raw_panel.quarter.astype(str))
periods_per_cell = raw_panel.groupby("grid_id").period.nunique()
missing_rows = int((len(expected_periods) - periods_per_cell).clip(lower=0).sum())
missing_periods = sorted(set(expected_periods) - set(raw_panel.period))
display(raw_panel.groupby("period").agg(Zeilen=("grid_id", "size"), Rasterzellen=("grid_id", "nunique")))
report("1.2", "OK" if duplicate_rows == 0 and not invalid_keys.any() and missing_rows == 0 else "PRÜFEN",
       f"Duplikate: {duplicate_rows:,}; ungültige Schlüssel: {invalid_keys.sum():,}; "
       f"fehlende Zellquartale: {missing_rows:,}; komplett fehlende Quartale: {missing_periods}.")

### 1.3 Nullwerte und positive Zählwerte (Pflicht)

In [ ]:
cell_totals = raw_panel.groupby("grid_id").births.sum()
display(raw_y.loc[raw_y > 0].describe(percentiles=[.5, .9, .95, .99]).to_frame("Positive Zählwerte"))
display(raw_y.value_counts().sort_index().rename("Häufigkeit").to_frame().head(25))
report("1.3", "HINWEIS", f"Nullanteil Zellquartale: {raw_y.eq(0).mean():.2%}; "
       f"Rasterzellen ohne jede Gründung im gesamten Zeitraum: {cell_totals.eq(0).mean():.2%}.")

### 1.4 Unbewohnte Rasterzellen mit Unternehmensstandorten (Pflicht)

In [ ]:
area_available = False
uninhabited_firm_ids = pd.Series([], dtype="string")
raster_path = RASTER_PATH
try:
    raster_path = RASTER_PATH
    area_columns = ["grid_id", "population_2019", "population_2025"]
    area = read_model_parquet(raster_path, columns=area_columns)
    firm_area = read_model_parquet(FIRMS_PATH, columns=["grid_id", "founding_date", "exit_date"])
    firm_area["founding_date"] = pd.to_datetime(firm_area.founding_date, errors="coerce")
    firm_area["exit_date"] = pd.to_datetime(firm_area.exit_date, errors="coerce")
    present = (firm_area.founding_date < pd.Timestamp(f"{END_YEAR+1}-01-01")) & (
        firm_area.exit_date.isna() | (firm_area.exit_date > pd.Timestamp(f"{START_YEAR}-01-01")))
    occupied_ids = firm_area.loc[present, "grid_id"].dropna().unique()
    no_population = area[["population_2019", "population_2025"]].eq(0).all(axis=1)
    uninhabited_firm_ids = area.loc[no_population & area.grid_id.isin(occupied_ids), "grid_id"]
    excluded_area_ids = set(uninhabited_firm_ids) - set(raw_panel.grid_id)
    report("1.4", "PRÜFEN" if excluded_area_ids else "OK",
           f"An beiden Bevölkerungsankern unbewohnt, mit Firmen im Zeitraum: {len(uninhabited_firm_ids):,} Zellen; "
           f"davon im Rohpanel ausgeschlossen: {len(excluded_area_ids):,}. Bereinigtes Modellsample folgt in 3.1.")
    print("Unbewohnt bezieht sich auf 2019/2025; historische Bewohnung ist damit nicht lückenlos nachgewiesen.")
    area_available = True
except (FileNotFoundError, ValueError, KeyError) as error:
    report("1.4", "OFFEN", f"Gebietsprüfung nicht auswertbar: {error}")

### 1.5 Verteilung und Ausreißer (Pflicht)

In [ ]:
display(raw_y.describe(percentiles=[.25, .5, .75, .9, .95, .99, .999]).to_frame("y"))
tail_cut = max(1.0, float(raw_y.quantile(.999)))
display(raw_panel.nlargest(10, "births")[["grid_id", "period", "births"]])
report("1.5", "HINWEIS", f"Mittelwert={raw_y.mean():.6g}; Varianz={raw_y.var():.6g}; "
       f"Maximum={raw_y.max():g}; Werte oberhalb max(1, 99.9%-Quantil): {(raw_y > tail_cut).sum():,}.")
print("Hohe Zählwerte werden angezeigt, aber nicht automatisch als Datenfehler entfernt.")

## Check required files and columns


In [ ]:
# Dieser Preflight benötigt keine firm_accessibility-Produkte und keine Spartenzuordnung.
for year in range(LAG_YEAR, END_YEAR + 1):
    year_dir = FEATURE_DIR / str(year)
    require_columns(year_dir / f"accessibility_potentials_{RESOLUTION}.parquet", [
        "grid_id", "year", "quarter", "period", "own_cell_pop", "own_cell_firms",
        "pop_access_15min", "existing_firms_access_15min",
    ])
    require_columns(year_dir / f"pedestrian_accessibility_quarter_{RESOLUTION}.parquet", [
        "grid_id", "year", "quarter", "period", "own_cell_walk_pop", "own_cell_walk_firms",
        "walk_pop_10min", "walk_firms_10min", "walk_pt_routes_10min", "pt_ohne_haltestelle",
    ])
    require_columns(year_dir / f"nearest_infrastructure_{RESOLUTION}.parquet", ["grid_id", "year", "tt_motorway_exit_min"])
print("Gründungs-Preflight bestanden: alle erforderlichen Jahre und Kernvariablen vorhanden.")

## Build the founding-model data


### Load and join the inputs


In [ ]:
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

panel_rows = con.execute(
    f"SELECT count(*) FROM read_parquet('{PANEL_PATH.as_posix()}') WHERE year BETWEEN {LAG_YEAR} AND {END_YEAR}"
).fetchone()[0]

founding = con.execute(f"""
SELECT p.grid_id, p.municipality_id, p.year, p.quarter, p.period,
       p.births, p.active_firms_tminus1,
       a.own_cell_pop, a.own_cell_firms,
       a.pop_access_15min, a.existing_firms_access_15min,
       w.own_cell_walk_pop, w.own_cell_walk_firms,
       w.walk_pop_10min, w.walk_firms_10min,
       w.walk_pt_routes_10min, w.pt_ohne_haltestelle,
       n.tt_motorway_exit_min
FROM read_parquet('{PANEL_PATH.as_posix()}') p
LEFT JOIN read_parquet('{(FEATURE_DIR/'*'/f'accessibility_potentials_{RESOLUTION}.parquet').as_posix()}') a
  USING (grid_id, year, quarter, period)
LEFT JOIN read_parquet('{(FEATURE_DIR/'*'/f'pedestrian_accessibility_quarter_{RESOLUTION}.parquet').as_posix()}') w
  USING (grid_id, year, quarter, period)
LEFT JOIN read_parquet('{(FEATURE_DIR/'*'/f'nearest_infrastructure_{RESOLUTION}.parquet').as_posix()}') n
  USING (grid_id, year)
WHERE p.year BETWEEN {LAG_YEAR} AND {END_YEAR}
ORDER BY p.grid_id, p.year, p.quarter
""").df()

if len(founding) != panel_rows:
    raise ValueError("The founding-data joins lost or duplicated panel rows.")
if founding.duplicated(["grid_id", "year", "quarter"]).any():
    raise ValueError("Duplicate cell-quarter rows after the joins.")

print(f"Panel input: {panel_rows:,} rows")
print(f"Joined founding data: {founding.shape[0]:,} rows × {founding.shape[1]} columns")
print(f"Cells: {founding['grid_id'].nunique():,}; quarters: {founding['period'].nunique()}")


### Remove incomplete routing rows


In [ ]:
founding_route_columns = [
    "pop_access_15min", "existing_firms_access_15min", "walk_pop_10min", "walk_firms_10min",
    "walk_pt_routes_10min", "pt_ohne_haltestelle", "tt_motorway_exit_min",
]


In [ ]:
raw_x_columns = ["active_firms_tminus1", "own_cell_pop", "own_cell_firms",
                 "own_cell_walk_pop", "own_cell_walk_firms", *founding_route_columns]
x_input_checks = pd.DataFrame({
    "NaN": founding[raw_x_columns].isna().sum(),
    "Unendlich": np.isinf(founding[raw_x_columns]).sum(),
    "Negativ": founding[raw_x_columns].lt(0).sum(),
})
display(x_input_checks)
bad_negative = founding[raw_x_columns].lt(0).any(axis=1)
bad_binary = ~founding.pt_ohne_haltestelle.isin([0, 1]) & founding.pt_ohne_haltestelle.notna()
inconsistent_rings = pd.Series(False, index=founding.index)
for total, own in [("pop_access_15min", "own_cell_pop"), ("existing_firms_access_15min", "own_cell_firms"),
                   ("walk_pop_10min", "own_cell_walk_pop"), ("walk_firms_10min", "own_cell_walk_firms")]:
    inconsistent_rings |= founding[total] + 1e-8 < founding[own]
if bad_negative.any() or bad_binary.any() or inconsistent_rings.any():
    raise ValueError(f"Unplausible X vor Transformation: negativ={bad_negative.sum()}, "
                     f"binär={bad_binary.sum()}, kumulative Masse < eigene Zelle={inconsistent_rings.sum()}.")

In [ ]:
incomplete = ~np.isfinite(founding[raw_x_columns]).all(axis=1)
print(f"Rows with incomplete routing: {incomplete.sum():,} ({incomplete.mean():.3%})")
founding = founding.loc[~incomplete].copy()
print(f"Rows retained: {len(founding):,}")

### Create neighbourhood variables


In [ ]:
# Non-overlapping neighbourhoods. The origin cell is removed.
founding["car_pop_ring_0_15"] = (founding["pop_access_15min"] - founding["own_cell_pop"]).clip(lower=0)
founding["car_firms_ring_0_15"] = (founding["existing_firms_access_15min"] - founding["own_cell_firms"]).clip(lower=0)
founding["walk_pop_ring_0_10"] = (founding["walk_pop_10min"] - founding["own_cell_walk_pop"]).clip(lower=0)
founding["walk_firms_ring_0_10"] = (founding["walk_firms_10min"] - founding["own_cell_walk_firms"]).clip(lower=0)

founding["log_pop_access_ring_0_15"] = np.log1p(founding["car_pop_ring_0_15"])
founding["log_firms_relative_car_ring_0_15"] = (
    np.log1p(founding["car_firms_ring_0_15"]) - founding["log_pop_access_ring_0_15"]
)
founding["log_walk_pop_ring_0_10"] = np.log1p(founding["walk_pop_ring_0_10"])
founding["log_firms_relative_walk_ring_0_10"] = (
    np.log1p(founding["walk_firms_ring_0_10"]) - founding["log_walk_pop_ring_0_10"]
)

print("Created car and walking population-mass and relative-firm-density variables.")


### Create population growth and lags


In [ ]:
founding = founding.sort_values(["grid_id", "year", "quarter"])
period_number = founding["year"] * 4 + founding["quarter"]
previous_period = period_number.groupby(founding["grid_id"], sort=False).shift(4)
previous_population = founding.groupby("grid_id", sort=False)["own_cell_pop"].shift(4)
previous_population = previous_population.where(period_number - previous_period == 4)

founding["log_own_firms"] = np.log1p(founding["active_firms_tminus1"])
founding["log_own_pop"] = np.log1p(previous_population)
founding["population_growth_yoy"] = np.log1p(founding["own_cell_pop"]) - np.log1p(previous_population)
founding["log_tt_motorway_exit"] = np.log1p(founding["tt_motorway_exit_min"])

before = len(founding)
founding = founding.loc[founding["year"] >= START_YEAR].copy()
print(f"Removed {before-len(founding):,} lead-in rows from {LAG_YEAR}.")


### Create the design matrix


In [ ]:
founding_terms = [
    "log_own_firms", "log_own_pop",
    "log_pop_access_ring_0_15", "log_firms_relative_car_ring_0_15",
    "log_walk_pop_ring_0_10", "log_firms_relative_walk_ring_0_10",
    "population_growth_yoy", "log_tt_motorway_exit",
    "walk_pt_routes_10min", "pt_ohne_haltestelle",
]

before = len(founding)
founding = founding.dropna(subset=founding_terms).copy()
period_dummies = pd.get_dummies(founding["period"], prefix="period", drop_first=True, dtype="float64")
X_founding = pd.concat([founding[founding_terms].astype("float64"), period_dummies], axis=1)
X_founding.insert(0, "const", 1.0)
y_founding = founding["births"].astype("float64")
founding_clusters = founding["grid_id"]

if not np.isfinite(X_founding.to_numpy()).all(): raise ValueError("Non-finite founding regressors.")
if not np.isfinite(y_founding).all() or (y_founding < 0).any() or (y_founding != np.floor(y_founding)).any():
    raise ValueError("Births must be non-negative integer counts.")

print(f"Rows removed for missing model values: {before-len(founding):,}")
print(f"Final founding sample: {len(founding):,} rows × {X_founding.shape[1]} design columns")
print(f"Births: {int(y_founding.sum()):,}; zero-count rows: {(y_founding==0).mean():.2%}")
print(f"Cell clusters: {founding_clusters.nunique():,}")


### Founding-model checks


In [ ]:
def correlation_report(frame, terms, threshold=0.70):
    corr = frame[terms].corr()
    pairs = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack()
    return pairs[pairs.abs() >= threshold].sort_values(key=abs, ascending=False).rename("correlation")

def condition_number(frame, terms):
    if frame[terms].nunique().le(1).any(): return np.inf
    eigenvalues = np.linalg.eigvalsh(frame[terms].corr().to_numpy())
    return np.inf if eigenvalues[0] <= 0 else float(np.sqrt(eigenvalues[-1] / eigenvalues[0]))


In [ ]:
founding_correlations = correlation_report(founding, founding_terms)
founding_summary = pd.Series({
    "rows": len(founding), "columns_in_design_matrix": X_founding.shape[1],
    "cells": founding["grid_id"].nunique(), "quarters": founding["period"].nunique(),
    "births": int(y_founding.sum()), "zero_share": float((y_founding==0).mean()),
    "birth_mean": float(y_founding.mean()), "birth_variance": float(y_founding.var()),
    "variance_to_mean": float(y_founding.var()/y_founding.mean()),
    "condition_number": condition_number(founding, founding_terms),
})
display(founding_summary.to_frame("value"))
print("Correlations with absolute value at least 0.70:")
print(founding_correlations.to_string() if len(founding_correlations) else "None")
founding_summary.to_csv(RESULT_DIR/"founding_input_diagnostics.csv", header=["value"])
founding_correlations.to_csv(RESULT_DIR/"founding_high_correlations.csv", header=True)
print("Saved 2 diagnostic files.")


## NB2-Checkliste: Spezifikation, Schätzung und Diagnostik

Die Zellen folgen hier den Rechenschritten: erst X prüfen, dann Poisson/NB2 schätzen,
dann Residuen und Robustheit untersuchen. Die Nummern entsprechen der Checkliste.
Die vollständigen Quartalsdummies kontrollieren gemeinsame Zeit-, Saison- und COVID-Effekte.
Ohne Offset wird die absolute Zahl der Gründungen modelliert.

### 3.1 Fehlende, unendliche, konstante und unplausible X (Pflicht)

In [ ]:
dropped_constants = [c for c in X_founding if c != "const" and X_founding[c].nunique() <= 1]
X_check = X_founding.drop(columns=dropped_constants).copy()
design_scale = X_check.std().replace(0, 1)
design_scale["const"] = 1.0
X_check = X_check / design_scale  # Nur Skalierung; identischer linearer Prädiktor.
remaining_bad = int((~np.isfinite(X_check)).sum().sum())
lost_cell_quarters = len(raw_panel) - len(founding)
area_retained = set(uninhabited_firm_ids) & set(founding.grid_id)
report("3.1", "OK" if remaining_bad == 0 and len(area_retained) == len(uninhabited_firm_ids) and not dropped_constants else "PRÜFEN",
       f"Modellzeilen={len(founding):,}; gegenüber Rohpanel entfernt={lost_cell_quarters:,}; "
       f"nichtendliche X übrig={remaining_bad}; entfernte konstante Spalten={dropped_constants}.")
if area_available: print(f"Unbewohnte Firmenzellen im Modellsample: {len(area_retained):,}/{len(uninhabited_firm_ids):,}.")
if area_available and len(area_retained) < len(uninhabited_firm_ids):
    print("PRÜFEN: Auch die Routing-/Lag-Bereinigung schließt unbewohnte Firmenzellen aus.")
display(founding.groupby("period").agg(Zeilen=("grid_id", "size"), Gründungen=("births", "sum")))
removed_constants_note = "Konstante Spalten werden nur für diese Diagnostik entfernt; Notebook 09 ggf. anpassen."
if dropped_constants: print(removed_constants_note)

### 3.2 Korrelationen, VIF und Rang (Pflicht)

In [ ]:
active_terms = [t for t in founding_terms if t in X_check]
display(founding[active_terms].corr().round(3))
# VIF aus der inversen vollständigen Korrelationsmatrix, inklusive Zeitdummies.
nonconstant = X_check.drop(columns="const")
correlation = nonconstant.corr().to_numpy()
eigenvalues = np.linalg.eigvalsh(correlation)
full_rank = bool(eigenvalues.min() > 1e-10)
vif = pd.Series(np.diag(np.linalg.inv(correlation)) if full_rank else np.inf,
                index=nonconstant.columns, name="VIF")
gram = X_check.to_numpy().T @ X_check.to_numpy()
design_rank = np.linalg.matrix_rank(gram, hermitian=True)
display(vif.sort_values(ascending=False).to_frame())
report("3.2", "PRÜFEN" if design_rank < X_check.shape[1] or vif.max() > 10 else "OK",
       f"Rang={design_rank}/{X_check.shape[1]}; maximaler VIF={vif.max():.3g}. "
       "Kumulative Auto-/Geh-Isochronen können sich trotz Ausschluss der eigenen Zelle überlappen.")
if design_rank < X_check.shape[1]:
    raise ValueError("Rangdefiziente Designmatrix vor der Modellschätzung bereinigen.")

### 3.4 Zeit-, Saison- und COVID-Effekte (Pflicht)

In [ ]:
display(founding.groupby(["year", "quarter"]).births.agg(["size", "sum", "mean"]))
time_columns = [c for c in X_check if c.startswith("period_")]
report("3.4", "OK" if len(time_columns) == founding.period.nunique()-1 else "PRÜFEN",
       f"{len(time_columns)} vollständige Quartals-Fixed-Effects plus Referenzquartal. "
       "Gemeinsame Jahres-, Saison- und COVID-Schocks sind darin enthalten.")
print("Separate COVID-/Jahres-/Saisondummies wären mit vollständigen Quartalsdummies kollinear.")

### 3.5 Offset-Entscheidung (Pflicht)

In [ ]:
report("3.5", "HINWEIS", "Zielgröße: absolute Gründungszahl je Rasterzelle und Quartal; kein Offset. "
       "Bevölkerung und Unternehmensbestand gehen mit frei geschätzten Koeffizienten ein.")
print("Für Gründungsraten wäre eine fachlich definierte, strikt positive Exposition nötig. "
      "log1p(Bestand) als Offset würde eine Rate pro (Bestand+1) erzwingen und ist keine neutrale Korrektur.")

### 3.6 Zeitliche Reihenfolge der Einflussgrößen (Pflicht)

In [ ]:
stock_mismatch = int((founding.own_cell_firms - founding.active_firms_tminus1).abs().gt(1e-8).sum())
timing = pd.DataFrame([
    ["Unternehmensbestand", "t-1 aus Panel; eigene Routingmasse gegen Panel geprüft"],
    ["log_own_pop", "Bevölkerung desselben Quartals im Vorjahr (shift(4), Lücken geprüft)"],
    ["population_growth_yoy", "Enthält Bevölkerung in t; keine strikt vorbestimmte Variable"],
    ["Erreichbare Bevölkerung", "Zeitgleiche zurückgerechnete Bevölkerung; Anker 2019/2025"],
    ["Erreichbare Unternehmen", "Notebook 06 aggregiert active_firms_tminus1"],
    ["Verkehrsnetz / ÖV", "Jährliche Snapshots; Stichtag innerhalb des Jahres nicht hier nachgewiesen"],
], columns=["Einflussgröße", "Messzeitpunkt / Einschränkung"])
display(timing)
report("3.6", "PRÜFEN" if stock_mismatch else "OFFEN",
       f"Abweichende eigene Firmenbestände: {stock_mismatch:,}. "
       "Historische Verfügbarkeit der Bevölkerungsanker/Netz-Snapshots extern prüfen; "
       "zeitgleiche Wachstumseffekte sind keine gesicherte zeitliche Ursache. Lag-Sensitivität in 5.3.")

### 2.1 Poisson-Basismodell mit denselben X (Pflicht)

In [ ]:
import warnings
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import NegativeBinomial

# Kleine Hilfsfunktion für die mehrfach benötigten NB-Schätzungen.
# L-BFGS begrenzt alpha numerisch auf [1e-6, 1e6]; Randtreffer werden markiert.
# Warnungen bleiben sichtbar und werden nicht als erfolgreiche Schätzung gewertet.
def fit_nb(y, X, groups, method="nb2", start=None, optimizer="lbfgs"):
    if start is None:
        p = sm.GLM(y, X, family=sm.families.Poisson()).fit(maxiter=200)
        start = np.r_[p.params, .1]
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        result = NegativeBinomial(y, X, loglike_method=method).fit(
            start_params=start, method=optimizer, maxiter=500, disp=0,
            cov_type="cluster", cov_kwds={"groups": groups},
            bounds=[(None, None)]*X.shape[1] + [(np.log(1e-6), np.log(1e6))])
    messages = sorted({str(w.message) for w in caught})
    for message in messages: print("Schätzwarnung:", message)
    return result, messages

poisson_check = sm.GLM(y_founding, X_check, family=sm.families.Poisson()).fit(
    maxiter=200, cov_type="cluster", cov_kwds={"groups": founding_clusters})
mu_poisson = np.asarray(poisson_check.predict(X_check))
report("2.1", "OK" if poisson_check.converged else "PRÜFEN",
       f"Poisson auf allen {len(y_founding):,} Modellzeilen und {X_check.shape[1]} X-Spalten; "
       f"Log-Likelihood={poisson_check.llf:,.3f}; konvergiert={poisson_check.converged}.")

### 2.2 Bedingte Überdispersion des Poisson-Modells (Pflicht)

In [ ]:
poisson_dispersion = float(np.sum((y_founding-mu_poisson)**2 / np.maximum(mu_poisson, 1e-12)) / poisson_check.df_resid)
report("2.2", "PRÜFEN" if poisson_dispersion > 1.2 or poisson_dispersion < .8 else "OK",
       f"Pearson-Dispersion={poisson_dispersion:.4f}; Abweichung von 1={poisson_dispersion-1:+.4f}. "
       "Werte deutlich über 1 sprechen für bedingte Überdispersion (Screen: >1.2).")
print("Die unbedingte Varianz/Mittelwert-Relation allein entscheidet nicht über NB2. "
      "Kein iid-Chi²-p-Wert bei abhängigen Panelbeobachtungen.")

### 2.3 NB2-Alpha und Abgrenzung zu Poisson (Pflicht)

In [ ]:
nb_check, nb_warnings = fit_nb(y_founding, X_check, founding_clusters,
                             start=np.r_[poisson_check.params, .1])
alpha = float(nb_check.params["alpha"])
alpha_ci = nb_check.conf_int().loc["alpha"]
ll_below_poisson = bool(nb_check.llf + .01 < poisson_check.llf)
lr_nb_poisson = max(0., 2*(nb_check.llf-poisson_check.llf)) if np.isfinite(nb_check.llf) else np.nan
boundary_p = 1.0 if lr_nb_poisson <= 1e-10 else .5 * stats.chi2.sf(lr_nb_poisson, 1)
report("2.3", "OK" if np.isfinite(nb_check.llf) and 1.01e-6 < alpha < .99e6 and alpha_ci.iloc[0] > 0 and nb_check.mle_retvals.get("converged", False) and not ll_below_poisson else "PRÜFEN",
       f"alpha={alpha:.6g}; clusterrobustes 95%-Wald-KI=[{alpha_ci.iloc[0]:.6g}, {alpha_ci.iloc[1]:.6g}]; "
       f"Boundary-LR={lr_nb_poisson:.3f}, iid-Referenz-p={boundary_p:.4g}.")
if ll_below_poisson:
    print("PRÜFEN: NB2-LogL liegt unter Poisson; mögliches Optimierungsproblem oder Poisson-Grenzfall.")
print("Numerische alpha-Grenzen: [1e-6, 1e6]; nahe der unteren Grenze ist NB2 praktisch Poisson. "
      "Ein Randtreffer ist kein Nachweis positiver Überdispersion.")
print("LR verwendet die 50:50-Randverteilung; bei Clusterabhängigkeit nur Referenz, "
      "kein clusterrobuster Signifikanznachweis. Wald-KI nahe alpha=0 vorsichtig interpretieren.")
if not np.isfinite(nb_check.llf) or not np.isfinite(alpha) or alpha <= 0 or not np.isfinite(nb_check.params).all():
    raise ValueError("Keine gültigen NB2-Parameter; nachfolgende Residuen wären unbrauchbar.")
mu_nb = np.asarray(nb_check.predict(X_check))
var_nb = mu_nb + alpha*mu_nb**2
pearson_nb = (y_founding.to_numpy()-mu_nb) / np.sqrt(np.maximum(var_nb, 1e-12))
diagnostics = founding[["grid_id", "municipality_id", "period", "year", "quarter", "births"]].copy()
diagnostics["mu_nb"] = mu_nb
diagnostics["mu_poisson"] = mu_poisson
diagnostics["pearson_nb"] = pearson_nb

### 2.4 Quadratische NB2-Varianzfunktion (Empfohlen)

In [ ]:
variance_screen = pd.DataFrame({"mu": mu_nb, "squared_error": (y_founding.to_numpy()-mu_nb)**2,
                                "NB2_Varianz": var_nb})
variance_screen["mu_Klasse"] = pd.qcut(variance_screen.mu, 10, duplicates="drop")
variance_bins = variance_screen.groupby("mu_Klasse", observed=True).agg(
    n=("mu", "size"), mu=("mu", "mean"), mittlerer_quadratischer_Fehler=("squared_error", "mean"),
    erwartete_NB2_Varianz=("NB2_Varianz", "mean"))
variance_bins["Fehler_zu_Varianz"] = variance_bins.mittlerer_quadratischer_Fehler / variance_bins.erwartete_NB2_Varianz
display(variance_bins)
nb_dispersion = float(np.sum(pearson_nb**2) / (len(y_founding)-len(nb_check.params)))
variance_flag = bool((variance_bins.Fehler_zu_Varianz.sub(1).abs() > .5).any())
report("2.4", "PRÜFEN" if variance_flag else "OK", f"NB2-Pearson-Dispersion={nb_dispersion:.4f}. "
       "Klassen zeigen E[(y-mu)²] gegenüber E[mu+alpha*mu²]; Abweichungen >50% werden markiert.")

### 2.5 Beobachtete und vorhergesagte Nullhäufigkeit (Pflicht)

In [ ]:
zero_probability_nb = np.exp(-np.log1p(alpha*mu_nb)/alpha)
observed_zero_share = float(y_founding.eq(0).mean())
zero_gap = observed_zero_share - float(zero_probability_nb.mean())
diagnostics["null_beobachtet"] = y_founding.eq(0).astype(int)
diagnostics["null_erwartet"] = zero_probability_nb
display(diagnostics.groupby("period")[["null_beobachtet", "null_erwartet"]].mean())
zero_flag = abs(zero_gap) > .01
report("2.5", "PRÜFEN" if zero_flag else "OK", f"Beobachtet={observed_zero_share:.3%}; "
       f"NB2-erwartet={zero_probability_nb.mean():.3%} ({zero_probability_nb.sum():,.1f} Nullzeilen); "
       f"Differenz={100*zero_gap:+.3f} Prozentpunkte (Screen: 1 Prozentpunkt).")
print("Eine hohe Nullhäufigkeit allein belegt keine Zero-Inflation; Differenzen auch nach Quartal prüfen.")

### 4.1 Konvergenz, Warnungen und stabile Schätzer (Pflicht)

In [ ]:
# Derselbe Optimizer mit verändertem Alpha-Start prüft die Schätzstabilität.
retry_start = nb_check.params.to_numpy().copy()
retry_start[-1] = max(float(retry_start[-1])*2, .05)
nb_retry, retry_warnings = fit_nb(y_founding, X_check, founding_clusters,
                                start=retry_start, optimizer="lbfgs")
fit_ok = bool(nb_check.mle_retvals.get("converged", False)) and np.isfinite(nb_check.bse).all() and np.isfinite(nb_check.llf)
retry_ok = bool(nb_retry.mle_retvals.get("converged", False)) and np.isfinite(nb_retry.params).all() and np.isfinite(nb_retry.llf)
ll_difference = abs(nb_retry.llf-nb_check.llf)
prediction_difference = float(np.max(np.abs(np.asarray(nb_retry.predict(X_check))-mu_nb)))
display(pd.DataFrame({"L-BFGS Start 1": nb_check.params, "L-BFGS Start 2": nb_retry.params}))
report("4.1", "OK" if fit_ok and retry_ok and ll_difference < .01 and not nb_warnings and not retry_warnings else "PRÜFEN",
       f"Start 1 konvergiert/SE endlich={fit_ok}; Start 2 konvergiert={retry_ok}; "
       f"|Delta LogL|={ll_difference:.6g}; max. |Delta Prognose|={prediction_difference:.6g}; "
       f"Warnungen={len(nb_warnings)+len(retry_warnings)}.")
if not fit_ok: print("PRÜFEN: Alle nachfolgenden Modellbefunde sind vorläufig, solange Konvergenz/SE unzureichend sind.")

### 4.2 Geeignete Standardfehler (Pflicht)

In [ ]:
# X_check = X_original / design_scale; deshalb beta_original = beta_check / scale.
parameter_scale = design_scale.reindex(nb_check.params.index).fillna(1.)
original_ci = nb_check.conf_int().div(parameter_scale, axis=0)
coefficients_original = pd.DataFrame({
    "Koeffizient_Originaleinheit": nb_check.params/parameter_scale,
    "SE_Zellcluster_Originaleinheit": nb_check.bse/parameter_scale,
    "KI95_unten": original_ci.iloc[:, 0], "KI95_oben": original_ci.iloc[:, 1],
    "Koeffizient_skaliertes_X": nb_check.params, "X_Divisor": parameter_scale,
})
display(coefficients_original)
print("Originaleinheit = Einheit der ursprünglichen Modellvariable (z.B. log1p), nicht Standardabweichung des jeweiligen Laufs. "
      "alpha bleibt ein Dispersionsparameter und wird nicht umskaliert.")
coefficients_original.to_csv(RESULT_DIR / "founding_nb2_diagnostic_coefficients.csv", index_label="term")
report("4.2", "HINWEIS", f"Poisson und NB2 verwenden Rasterzellen-Cluster: {founding_clusters.nunique():,}. "
       "Damit ist Abhängigkeit innerhalb derselben Zelle über die Zeit zugelassen. "
       "Abhängigkeit zwischen benachbarten Zellen bleibt offen; siehe Moran-Test in 4.4.")

### 4.3 Große Residuen und einflussreiche Rasterzellen (Empfohlen)

In [ ]:
diagnostics["abs_residuum"] = np.abs(pearson_nb)
display(diagnostics.nlargest(10, "abs_residuum")[["grid_id", "period", "births", "mu_nb", "pearson_nb"]])
influential_cells = diagnostics.groupby("grid_id").agg(
    Gründungen=("births", "sum"), Residuenquadrate=("pearson_nb", lambda s: float(np.sum(s**2))))
display(influential_cells.nlargest(10, "Residuenquadrate"))
report("4.3", "PRÜFEN" if np.max(np.abs(pearson_nb)) > 5 else "OK",
       f"|Pearson-Residuum|>3: {np.sum(np.abs(pearson_nb)>3):,}; Maximum={np.max(np.abs(pearson_nb)):.3g}. "
       "Residuenrangliste ist ein Einfluss-Screen, keine Cook-Distanz; Ausschluss-Refit in 5.3.")

### 4.4 Räumliche Residuenautokorrelation: Moran’s I (Pflicht)

In [ ]:
spatial_flag = False
try:
    from scipy.spatial import cKDTree
    from scipy.sparse import csr_matrix

    coordinates = pd.read_parquet(raster_path, columns=["grid_id", "centroid_x", "centroid_y"])
    moran_rows = []
    rng = np.random.default_rng(RANDOM_SEED)
    for period, group in diagnostics.groupby("period", sort=True):
        points = group[["grid_id", "pearson_nb"]].merge(coordinates, on="grid_id", validate="one_to_one")
        if points[["centroid_x", "centroid_y"]].isna().any().any():
            raise ValueError("Fehlende Rasterkoordinaten für Moran's I.")
        pairs = cKDTree(points[["centroid_x", "centroid_y"]]).query_pairs(CELL_SIZE*np.sqrt(2)+.01, output_type="ndarray")
        n = len(points)
        if len(pairs) == 0 or n < 4:
            moran_rows.append({"period": period, "n": n, "I": np.nan, "p_perm": np.nan, "Inseln": n})
            continue
        W = csr_matrix((np.ones(2*len(pairs)), (np.r_[pairs[:, 0], pairs[:, 1]], np.r_[pairs[:, 1], pairs[:, 0]])), shape=(n, n))
        degrees = np.asarray(W.sum(axis=1)).ravel()
        W = W.multiply((1/np.maximum(degrees, 1))[:, None]).tocsr()
        z = points.pearson_nb.to_numpy() - points.pearson_nb.mean()
        denom = z @ z
        if denom <= 0:
            moran_rows.append({"period": period, "n": n, "I": np.nan, "p_perm": np.nan, "Inseln": int((degrees == 0).sum())})
            continue
        factor = n / W.sum()
        I = float(factor * (z @ (W @ z)) / denom)
        expected_I = -1/(n-1)
        extreme = 0
        for permutation in range(MORAN_PERMUTATIONS):
            zp = rng.permutation(z)
            simulated = float(factor * (zp @ (W @ zp)) / denom)
            extreme += abs(simulated-expected_I) >= abs(I-expected_I)
        moran_rows.append({"period": period, "n": n, "I": I,
                           "p_perm": (extreme+1)/(MORAN_PERMUTATIONS+1), "Inseln": int((degrees == 0).sum())})
    moran_results = pd.DataFrame(moran_rows)
    display(moran_results)
    # Holm-Korrektur für die gemeinsame Betrachtung aller Quartale.
    valid_p = moran_results.p_perm.dropna().sort_values()
    holm = np.maximum.accumulate(valid_p.to_numpy() * np.arange(len(valid_p), 0, -1))
    moran_results["p_Holm"] = pd.Series(np.minimum(holm, 1), index=valid_p.index)
    spatial_flag = bool(moran_results.p_Holm.lt(.05).any())
    display(moran_results[["period", "I", "p_perm", "p_Holm", "Inseln"]])
    report("4.4", "PRÜFEN" if spatial_flag else ("OK" if len(valid_p) == len(moran_results) else "OFFEN"),
           f"Moran's I je Quartal, Queen-Nachbarn, zeilenstandardisiert; {MORAN_PERMUTATIONS} Permutationen; "
           f"Holm-p<.05 in {moran_results.p_Holm.lt(.05).sum()} Quartalen. SLX garantiert keine unabhängigen Residuen.")
    print("Permutationstest als Residuen-Screen: Heteroskedastizität/Parameterschätzung beeinflussen die Referenz. "
          "Inseln ohne Nachbarn werden ausgewiesen.")
except (FileNotFoundError, ValueError, KeyError) as error:
    report("4.4", "OFFEN", f"Räumliche Prüfung nicht auswertbar: {error}")

### 4.5 Zeitliche Residuenabhängigkeit (Pflicht)

In [ ]:
ordered = diagnostics.sort_values(["grid_id", "year", "quarter"]).copy()
ordered["t"] = ordered.year*4 + ordered.quarter
ordered["residuum_tminus1"] = ordered.groupby("grid_id").pearson_nb.shift()
adjacent = ordered.t - ordered.groupby("grid_id").t.shift() == 1
adjacent_rows = ordered.loc[adjacent].dropna(subset=["residuum_tminus1"])
serial_rho = adjacent_rows.pearson_nb.corr(adjacent_rows.residuum_tminus1)
display(adjacent_rows.groupby("period")[["pearson_nb", "residuum_tminus1"]].corr().iloc[0::2, 1].rename("Lag1_Korrelation"))
report("4.5", "PRÜFEN" if np.isfinite(serial_rho) and abs(serial_rho) > .1 else ("OK" if np.isfinite(serial_rho) else "OFFEN"),
       f"Lag-1-Korrelation nur tatsächlich aufeinanderfolgender Zellquartale: {serial_rho:.4f}; "
       f"Paare={len(adjacent_rows):,} (Screen: |rho|>.1). Zellcluster-SE berücksichtigen diese Abhängigkeit.")

### 4.6 Vorhersagen nach Größenklassen, Quartalen und Teilräumen (Empfohlen)

In [ ]:
diagnostics["Größenklasse"] = pd.cut(diagnostics.births, [-1, 0, 1, 2, 5, np.inf], labels=["0", "1", "2", "3–5", ">5"])
for grouping in ["Größenklasse", "period", "municipality_id"]:
    calibration = diagnostics.groupby(grouping, observed=True).agg(
        n=("births", "size"), beobachtet=("births", "sum"), NB2=("mu_nb", "sum"), Poisson=("mu_poisson", "sum"))
    calibration["NB2_minus_beobachtet"] = calibration.NB2-calibration.beobachtet
    print("Gruppierung:", grouping)
    display(calibration if grouping != "municipality_id" else calibration.reindex(
        calibration.NB2_minus_beobachtet.abs().sort_values(ascending=False).head(20).index))
report("4.6", "HINWEIS", "In-Sample-Kalibrierung ausgegeben; Teilräume: Gemeinden (20 größte absolute Fehler). "
       "Klassen nach beobachtetem y sind deskriptiv und kein unabhängiger Gütenachweis.")

### 5.1 Poisson und NB2 vergleichen (Pflicht)

In [ ]:
comparison = []
for name, result, predicted, dispersion in [
    ("Poisson", poisson_check, mu_poisson, poisson_dispersion),
    ("NB2", nb_check, mu_nb, nb_dispersion)]:
    comparison.append({"Modell": name, "n": len(y_founding), "AIC": result.aic, "LogL": result.llf,
                       "Pearson_Dispersion": dispersion,
                       "MAE_in_sample": float(np.mean(np.abs(y_founding-predicted))),
                       "RMSE_in_sample": float(np.sqrt(np.mean((y_founding-predicted)**2)))})
model_comparison = pd.DataFrame(comparison).set_index("Modell")
display(model_comparison)
report("5.1", "HINWEIS", f"NB2 minus Poisson AIC={nb_check.aic-poisson_check.aic:+.3f}; "
       "gleiche Zeilen, X und Offset-Entscheidung. AIC ist bei Clusterabhängigkeit ein deskriptiver Vergleich.")

### Zusätzliche Modelltests auf einer festen Rasterzellen-Stichprobe

Die folgenden Re-Fits nutzen höchstens `ROBUSTNESS_CELLS` zufällig ausgewählte Zellen
mit allen verfügbaren Quartalen (fester Seed). Sie sind Sensitivitätsscreens und ersetzen
die vollständige Schätzung oben nicht. Jede Variante meldet n, Konvergenz und Warnungen.

In [ ]:
rng_sample = np.random.default_rng(RANDOM_SEED)
sample_ids = rng_sample.choice(founding.grid_id.unique(),
                               size=min(ROBUSTNESS_CELLS, founding.grid_id.nunique()), replace=False)
sensitivity = founding.loc[founding.grid_id.isin(sample_ids)].copy()
X_sensitivity = X_check.loc[sensitivity.index].copy()
y_sensitivity = y_founding.loc[sensitivity.index]
nb_sensitivity, sample_warnings = fit_nb(y_sensitivity, X_sensitivity, sensitivity.grid_id, start=nb_check.params.to_numpy())
print(f"Zusatztests: {len(sample_ids):,} Rasterzellen; {len(sensitivity):,} Zeilen; "
      f"Basis konvergiert={nb_sensitivity.mle_retvals.get('converged', False)}; Warnungen={len(sample_warnings)}.")

### 3.3 Funktionale Form im Log-Erwartungswert (Empfohlen)

In [ ]:
form_flag = False
try:
    form_rows = []
    # Je Variable ein zusätzlicher quadratischer Term; ein einfacher Nichtlinearitäts-Screen.
    for term in ["log_own_firms", "log_own_pop", "log_pop_access_ring_0_15", "log_tt_motorway_exit"]:
        if term not in X_sensitivity: continue
        extended = X_sensitivity.copy()
        centered = extended[term]-extended[term].mean()
        extended[term+"_quadratisch"] = centered**2
        result, messages = fit_nb(y_sensitivity, extended, sensitivity.grid_id,
            start=np.r_[nb_sensitivity.params.iloc[:-1], 0., nb_sensitivity.params.iloc[-1]])
        form_rows.append({"Variable": term, "n": len(extended), "Delta_AIC": result.aic-nb_sensitivity.aic,
                          "p_quadratisch_cluster": result.pvalues[term+"_quadratisch"],
                          "konvergiert": result.mle_retvals.get("converged", False), "Warnungen": len(messages)})
    form_results = pd.DataFrame(form_rows)
    display(form_results)
    form_flag = bool(form_results.p_quadratisch_cluster.lt(.05/max(len(form_results), 1)).any())
    report("3.3", "PRÜFEN" if form_flag or not form_results.konvergiert.all() or form_results.Warnungen.gt(0).any() else "HINWEIS",
           "Quadratische Erweiterungen mit clusterrobustem Wald-Test; Bonferroni-Screen über getestete Variablen. "
           "Bei Auffälligkeiten gezielt Splines/Transformationen prüfen; kein vollständiger Linearitätsnachweis.")
except (ValueError, RuntimeError, np.linalg.LinAlgError, OverflowError) as error:
    report("3.3", "OFFEN", f"Zusatzmodell nicht auswertbar: {type(error).__name__}: {error}")


### 5.2 Zeitlich getrennte Validierung (Empfohlen)

In [ ]:
try:
    test_year = int(sensitivity.year.max())
    train = sensitivity.year < test_year
    # Vollständige Perioden-FE lassen sich nicht in unbekannte Zukunftsquartale extrapolieren.
    # Daher gesonderte prognosefähige Spezifikation mit Trend, Saison und COVID-Indikator.
    validation_X = sensitivity[active_terms].copy()
    validation_X["const"] = 1.
    validation_X["Jahrestrend"] = sensitivity.year-START_YEAR
    for q in [2, 3, 4]: validation_X[f"Saison_Q{q}"] = sensitivity.quarter.eq(q).astype(float)
    validation_X["COVID_2020_2021"] = sensitivity.year.between(2020, 2021).astype(float)
    validation_X = validation_X.loc[:, validation_X.loc[train].nunique() > 1].copy().assign(const=1.)
    training_scale = validation_X.loc[train].std().replace(0, 1)
    training_scale["const"] = 1.
    validation_X = validation_X/training_scale
    validation_rows = []
    if train.any() and (~train).any() and sensitivity.loc[train].grid_id.nunique() > 1:
        p_train = sm.GLM(y_sensitivity.loc[train], validation_X.loc[train], family=sm.families.Poisson()).fit()
        nb_train, validation_warnings = fit_nb(y_sensitivity.loc[train], validation_X.loc[train], sensitivity.loc[train, "grid_id"])
        for name, result in [("Poisson", p_train), ("NB2", nb_train)]:
            pred = np.asarray(result.predict(validation_X.loc[~train]))
            actual = y_sensitivity.loc[~train].to_numpy()
            if name == "NB2":
                a = float(result.params["alpha"])
                log_prob = stats.nbinom.logpmf(actual, 1/a, 1/(1+a*pred))
            else:
                log_prob = stats.poisson.logpmf(actual, pred)
            validation_rows.append({"Modell": name, "Testjahr": test_year, "n_train": int(train.sum()),
                "n_test": int((~train).sum()), "MAE": np.mean(np.abs(actual-pred)),
                "RMSE": np.sqrt(np.mean((actual-pred)**2)), "mittlerer_LogScore": np.mean(log_prob),
                "beobachtet": actual.sum(), "prognostiziert": pred.sum()})
        display(pd.DataFrame(validation_rows))
        print("NB2-Training konvergiert:", nb_train.mle_retvals.get("converged", False))
        report("5.2", "HINWEIS" if nb_train.mle_retvals.get("converged", False) and not validation_warnings else "PRÜFEN", f"Trainiert bis {test_year-1}, getestet {test_year}; "
               "gesonderte Trend-/Saisonspezifikation, Skalierung nur aus Training. "
               "Bedingte Prognose mit realisierten Test-X; keine echte ex-ante-Prognose.")
    else:
        report("5.2", "OFFEN", "Zu wenig Zeit- oder Clusterunterstützung für zeitliche Validierung.")
except (ValueError, RuntimeError, np.linalg.LinAlgError, OverflowError) as error:
    report("5.2", "OFFEN", f"Zusatzmodell nicht auswertbar: {type(error).__name__}: {error}")


### 5.3 Schwellen, Nachbarschaft, Lags und einflussreiche Zellen (Empfohlen)

In [ ]:
try:
    sensitivity_rows = []
    def compare_variant(name, X_variant, frame=sensitivity):
        keep = np.isfinite(X_variant).all(axis=1)
        Xv = X_variant.loc[keep]
        if len(Xv) <= Xv.shape[1] or frame.loc[Xv.index, "grid_id"].nunique() < 2:
            print(name, "OFFEN: zu wenig gültige Beobachtungen")
            return
        try:
            yv = frame.loc[Xv.index, "births"]
            groups = frame.loc[Xv.index, "grid_id"]
            # Bei geänderter Zeilenauswahl beide Modelle auf derselben Teilmenge schätzen.
            base_X = X_sensitivity.loc[Xv.index].copy()
            # Nach Zeilenausschlüssen eine tatsächlich beobachtete Referenzperiode wählen.
            drop_columns = [c for c in Xv if c != "const" and Xv[c].nunique() <= 1]
            reference_column = "period_" + str(frame.loc[Xv.index, "period"].min())
            if reference_column in Xv: drop_columns.append(reference_column)
            Xv = Xv.drop(columns=drop_columns)
            base_X = base_X.drop(columns=drop_columns)
            base = nb_sensitivity
            if not Xv.index.equals(X_sensitivity.index) or list(Xv.columns) != list(X_sensitivity.columns):
                base, _ = fit_nb(yv, base_X, groups, start=nb_sensitivity.params.reindex([*base_X.columns, "alpha"]).to_numpy())
            variant, messages = fit_nb(yv, Xv, groups, start=base.params.to_numpy())
            sensitivity_rows.append({"Variante": name, "n": len(Xv), "Delta_AIC_gleiche_Zeilen": variant.aic-base.aic,
                "alpha": variant.params["alpha"], "max_Delta_Beta_skaliert": float(np.max(np.abs(variant.params.drop("alpha")-base.params.drop("alpha")))),
                "konvergiert": variant.mle_retvals.get("converged", False), "Warnungen": len(messages)})
        except (ValueError, RuntimeError, np.linalg.LinAlgError, OverflowError) as error:
            print(name, "OFFEN:", str(error))
            sensitivity_rows.append({"Variante": name, "n": len(Xv), "konvergiert": False, "Fehler": str(error)})
    
    
    for minutes in [10, 30]:
        columns = ["grid_id", "year", "quarter", f"pop_access_{minutes}min", f"existing_firms_access_{minutes}min"]
        feature_paths = [FEATURE_DIR/str(y)/f"accessibility_potentials_{RESOLUTION}.parquet" for y in sensitivity.year.unique()]
        if not all(set(columns).issubset(pq.ParquetFile(p).schema_arrow.names) for p in feature_paths):
            print(f"{minutes}-Minuten-Variante OFFEN: erforderliche Features fehlen.")
            continue
        extra = pd.concat([pd.read_parquet(p, columns=columns, filters=[("grid_id", "in", list(sample_ids))]) for p in feature_paths])
        extra = sensitivity[["grid_id", "year", "quarter", "own_cell_pop", "own_cell_firms"]].merge(
            extra, on=["grid_id", "year", "quarter"], how="left", validate="one_to_one")
        extra.index = sensitivity.index
        changed = X_sensitivity.copy()
        pop = np.log1p((extra[f"pop_access_{minutes}min"]-extra.own_cell_pop).clip(lower=0))
        firms = np.log1p((extra[f"existing_firms_access_{minutes}min"]-extra.own_cell_firms).clip(lower=0))
        changed["log_pop_access_ring_0_15"] = pop/design_scale["log_pop_access_ring_0_15"]
        changed["log_firms_relative_car_ring_0_15"] = (firms-pop)/design_scale["log_firms_relative_car_ring_0_15"]
        compare_variant(f"Auto-Isochrone {minutes} statt 15 Minuten", changed)
    
    changed = X_sensitivity.copy()
    changed["log_pop_access_ring_0_15"] = np.log1p(sensitivity.pop_access_15min)/design_scale["log_pop_access_ring_0_15"]
    changed["log_firms_relative_car_ring_0_15"] = (np.log1p(sensitivity.existing_firms_access_15min)-np.log1p(sensitivity.pop_access_15min))/design_scale["log_firms_relative_car_ring_0_15"]
    compare_variant("Auto-Nachbarschaft inklusive eigener Zelle", changed)
    
    changed = X_sensitivity.copy()
    lagged = sensitivity.sort_values(["grid_id", "year", "quarter"])
    t = lagged.year*4+lagged.quarter
    consecutive = t-t.groupby(lagged.grid_id).shift() == 1
    lag_terms = [c for c in active_terms if c != "log_own_firms"]
    changed.loc[lagged.index, lag_terms] = X_sensitivity.loc[lagged.index, lag_terms].groupby(lagged.grid_id).shift().where(consecutive, np.nan, axis=0)
    compare_variant("Zusätzliches Quartalslag der übrigen zeitvariablen X", changed)
    
    top_ids = diagnostics.loc[diagnostics.grid_id.isin(sample_ids)].groupby("grid_id").abs_residuum.max().nlargest(5).index
    compare_variant("Ohne fünf Zellen mit größten Residuen", X_sensitivity.loc[~sensitivity.grid_id.isin(top_ids)])
    sensitivity_results = pd.DataFrame(sensitivity_rows)
    display(sensitivity_results)
    report("5.3", "PRÜFEN" if not sensitivity_results.konvergiert.all() or sensitivity_results.get("Warnungen", pd.Series(dtype=float)).gt(0).any() else "HINWEIS", "Sensitivität auf fester Zellstichprobe; AIC-Differenzen jeweils auf identischen Zeilen. "
           "Maximale Koeffizientänderung ist ein Screen; Schwellenvariablen besitzen eine veränderte inhaltliche Bedeutung.")
except (ValueError, RuntimeError, np.linalg.LinAlgError, OverflowError) as error:
    report("5.3", "OFFEN", f"Zusatzmodell nicht auswertbar: {type(error).__name__}: {error}")


### 5.4 Alternativen bei Fehlspezifikation (Bedingt)

In [ ]:
try:
    from statsmodels.discrete.count_model import ZeroInflatedNegativeBinomialP
    
    alternatives_needed = variance_flag or zero_flag or form_flag
    if alternatives_needed:
        alternative_rows = [{"Modell": "NB2", "AIC": nb_sensitivity.aic, "LogL": nb_sensitivity.llf,
                             "konvergiert": nb_sensitivity.mle_retvals.get("converged", False), "Warnungen": len(sample_warnings)}]
        nb1, nb1_warnings = fit_nb(y_sensitivity, X_sensitivity, sensitivity.grid_id, method="nb1")
        alternative_rows.append({"Modell": "NB1", "AIC": nb1.aic, "LogL": nb1.llf,
                                 "konvergiert": nb1.mle_retvals.get("converged", False), "Warnungen": len(nb1_warnings)})
        if zero_flag:
            zinb_start = np.r_[0., nb_sensitivity.params]
            zinb_start[1 + X_sensitivity.columns.get_loc("const")] += np.log(2)
            zinb_start[-1] = min(.5, float(nb_sensitivity.params["alpha"]))
            with warnings.catch_warnings(record=True) as zinb_caught:
                warnings.simplefilter("always")
                zinb = ZeroInflatedNegativeBinomialP(y_sensitivity, X_sensitivity,
                    exog_infl=np.ones((len(sensitivity), 1)), p=2).fit(
                    start_params=zinb_start, method="lbfgs", maxiter=500, disp=0,
                    bounds=[(None, None)]*(X_sensitivity.shape[1]+1) + [(1e-6, 1e6)],
                    cov_type="cluster", cov_kwds={"groups": sensitivity.grid_id})
            zinb_messages = sorted({str(w.message) for w in zinb_caught})
            for message in zinb_messages: print("ZINB-Warnung:", message)
            alternative_rows.append({"Modell": "ZINB (Inflation: Konstante)", "AIC": zinb.aic, "LogL": zinb.llf,
                                     "konvergiert": zinb.mle_retvals.get("converged", False), "Warnungen": len(zinb_messages)})
        display(pd.DataFrame(alternative_rows))
        report("5.4", "PRÜFEN", "Fehlspezifikationsscreen auffällig: NB1 und bei Nullabweichung ZINB getestet. "
               "Poisson mit Zellcluster-SE steht in 5.1. Alternativen nur bei gültiger Konvergenz interpretieren; "
               "räumliche/zeitliche Abhängigkeit wird durch NB1/ZINB nicht behoben.")
    else:
        report("5.4", "HINWEIS", "Kein Auslöser in den Varianz-, Null- und Form-Screens; "
               "keine zusätzlichen NB1/ZINB-Fits erforderlich. Poisson mit robusten SE bereits geschätzt.")
except (ValueError, RuntimeError, np.linalg.LinAlgError, OverflowError) as error:
    report("5.4", "OFFEN", f"Zusatzmodell nicht auswertbar: {type(error).__name__}: {error}")


## Build the survival-model data


In [ ]:
# Erst nach den NB2-Prüfungen: fehlende Survival-Produkte blockieren die Checkliste nicht.
survival_available = True
survival_status = "OK"
survival_messages = []
survival_requirements = [(FIRMS_PATH, [
    "firm_id", "founding_date", "exit_date", "exit_observed", "Sparte_ID", "Sparte_Text",
])]
for year in range(START_YEAR, END_YEAR + 1):
    survival_requirements.append((FEATURE_DIR / str(year) / f"firm_accessibility_quarter_{RESOLUTION}.parquet", [
        "firm_id", "grid_id", "Fachgruppe_ID", "year", "quarter", "period",
        "included_in_lagged_stock", "own_cell_pop", "own_cell_firms",
        "own_cell_same_fachgruppe_firms", "pop_access_15min",
        "existing_firms_access_15min", "same_fachgruppe_firms_access_15min",
    ]))
for path, required in survival_requirements:
    try:
        require_columns(path, required)
    except (FileNotFoundError, ValueError) as error:
        survival_available = False
        survival_status = "OFFEN"
        survival_messages.append(str(error))
if survival_available:
    # Eine In-Memory-View normalisiert den historischen Schlüssel vor den Joins.
    parts = []
    for year in range(START_YEAR, END_YEAR + 1):
        path = FEATURE_DIR / str(year) / f"firm_accessibility_quarter_{RESOLUTION}.parquet"
        actual_grid = grid_column(path)
        projection = "*" if actual_grid == "grid_id" else "* EXCLUDE (grid_id_100m), grid_id_100m AS grid_id"
        sql_path = path.as_posix().replace("'", "''")
        parts.append(f"SELECT {projection} FROM read_parquet('{sql_path}')")
    con.execute("CREATE OR REPLACE TEMP VIEW survival_features AS " + " UNION ALL BY NAME ".join(parts))
    print("Überlebens-Preflight bestanden; Schlüssel in der In-Memory-View vereinheitlicht.")
else:
    print("Überlebensprüfungen OFFEN; NB2-Ergebnisse bleiben verfügbar:")
    for message in survival_messages: print(message)

### Load and join firm-quarter inputs


In [ ]:
if survival_available:
    try:
        raw_survival_rows = con.execute(
            f"SELECT count(*) FROM survival_features WHERE year BETWEEN {START_YEAR} AND {END_YEAR}"
        ).fetchone()[0]

        spells = con.execute(f"""
        SELECT a.firm_id AS standort_id, a.grid_id,
               CAST(a.Fachgruppe_ID AS VARCHAR) AS Fachgruppe_ID,
               a.year, a.quarter, a.period, a.included_in_lagged_stock,
               a.own_cell_pop, a.own_cell_firms, a.own_cell_same_fachgruppe_firms,
               a.pop_access_15min, a.existing_firms_access_15min,
               a.same_fachgruppe_firms_access_15min,
               w.walk_pt_routes_10min, w.pt_ohne_haltestelle,
               n.tt_motorway_exit_min
        FROM survival_features a
        JOIN read_parquet('{(FEATURE_DIR/'*'/f'pedestrian_accessibility_quarter_{RESOLUTION}.parquet').as_posix()}') w
          ON a.grid_id=w.grid_id AND a.year=w.year AND a.quarter=w.quarter
        JOIN read_parquet('{(FEATURE_DIR/'*'/f'nearest_infrastructure_{RESOLUTION}.parquet').as_posix()}') n
          ON a.grid_id=n.grid_id AND a.year=n.year
        WHERE a.year BETWEEN {START_YEAR} AND {END_YEAR}
        ORDER BY a.firm_id, a.year, a.quarter
        """).df()

        if len(spells) != raw_survival_rows: raise ValueError("The survival joins lost or duplicated rows.")
        if spells.duplicated(["standort_id", "year", "quarter"]).any(): raise ValueError("Duplicate firm-quarter rows.")
        print(f"Firm-quarter input: {raw_survival_rows:,} rows")
        print(f"Joined survival data: {spells.shape[0]:,} rows × {spells.shape[1]} columns")
        print(f"Locations: {spells['standort_id'].nunique():,}")
    except (FileNotFoundError, ValueError, KeyError, np.linalg.LinAlgError) as error:
        survival_available = False
        survival_status = "PRÜFEN"
        survival_messages.append(str(error))
        print(f"Überlebensprüfung PRÜFEN: {error}")
else:
    print("Überlebens-Prüfschritt nicht ausgeführt; siehe separaten Survival-Preflight/Befund.")

### Remove incomplete firm histories


In [ ]:
if survival_available:
    try:
        survival_route_columns = [
            "pop_access_15min", "existing_firms_access_15min", "same_fachgruppe_firms_access_15min",
            "walk_pt_routes_10min", "pt_ohne_haltestelle", "tt_motorway_exit_min",
        ]
        bad_ids = spells.loc[spells[survival_route_columns].isna().any(axis=1), "standort_id"].unique()
        missing_group_ids = spells.loc[spells["Fachgruppe_ID"].isna(), "standort_id"].unique()
        excluded_ids = np.union1d(bad_ids, missing_group_ids)
        spells = spells.loc[~spells["standort_id"].isin(excluded_ids)].copy()
        print(f"Locations removed with incomplete histories: {len(excluded_ids):,}")
        print(f"Rows retained: {len(spells):,}; locations retained: {spells['standort_id'].nunique():,}")

    except (FileNotFoundError, ValueError, KeyError, np.linalg.LinAlgError) as error:
        survival_available = False
        survival_status = "PRÜFEN"
        survival_messages.append(str(error))
        print(f"Überlebensprüfung PRÜFEN: {error}")
else:
    print("Überlebens-Prüfschritt nicht ausgeführt; siehe separaten Survival-Preflight/Befund.")

### Create survival intervals


In [ ]:
if survival_available:
    try:
        firm_dates = pd.read_parquet(FIRMS_PATH, columns=[
            "firm_id", "founding_date", "exit_date", "exit_observed", "Sparte_ID", "Sparte_Text",
        ]).rename(columns={"firm_id":"standort_id", "Sparte_ID":"sparte", "Sparte_Text":"sparte_name"})
        if firm_dates["standort_id"].duplicated().any(): raise ValueError("firm_id is not unique.")
        firm_dates["founding_date"] = pd.to_datetime(firm_dates["founding_date"], errors="coerce")
        firm_dates["exit_date"] = pd.to_datetime(firm_dates["exit_date"], errors="coerce")
        firm_dates["sparte"] = firm_dates["sparte"].astype("string")
        spells = spells.merge(firm_dates, on="standort_id", how="left", validate="many_to_one")
        if spells[["founding_date", "sparte", "sparte_name"]].isna().any().any():
            raise ValueError("A routed location lacks dates or an official sector mapping.")

        quarter_index = pd.PeriodIndex(spells["period"], freq="Q").asi8
        founding_index = spells["founding_date"].dt.to_period("Q").array.asi8
        spells["start"] = quarter_index - founding_index
        spells["stop"] = spells["start"] + 1
        exit_quarter = spells["exit_date"].dt.to_period("Q").astype("string")
        spells["event"] = (spells["exit_observed"].fillna(False) & spells["period"].eq(exit_quarter)).astype("int8")
        spells = spells.sort_values(["standort_id", "year", "quarter"])

        if (spells["start"] < 0).any(): raise ValueError("A survival interval starts before firm founding.")
        if (spells.groupby("standort_id")["event"].sum() > 1).any(): raise ValueError("A location has multiple exits.")
        sequence = spells["year"]*4 + spells["quarter"]
        if sequence.groupby(spells["standort_id"]).diff().dropna().ne(1).any(): raise ValueError("Firm histories contain gaps.")

        entry_age = spells.groupby("standort_id")["start"].min()
        print(f"Observed exits: {int(spells['event'].sum()):,}")
        print(f"Left-truncated locations: {(entry_age>0).sum():,} ({(entry_age>0).mean():.2%})")

    except (FileNotFoundError, ValueError, KeyError, np.linalg.LinAlgError) as error:
        survival_available = False
        survival_status = "PRÜFEN"
        survival_messages.append(str(error))
        print(f"Überlebensprüfung PRÜFEN: {error}")
else:
    print("Überlebens-Prüfschritt nicht ausgeführt; siehe separaten Survival-Preflight/Befund.")

### Create Cox covariates


In [ ]:
if survival_available:
    try:
        focal = spells["included_in_lagged_stock"].astype("float64")
        spells["own_firms"] = (spells["own_cell_firms"] - focal).clip(lower=0)
        spells["own_same"] = (spells["own_cell_same_fachgruppe_firms"] - focal).clip(lower=0)
        spells["own_other"] = (spells["own_firms"] - spells["own_same"]).clip(lower=0)

        all_firms = (spells["existing_firms_access_15min"] - focal).clip(lower=0)
        same_firms = (spells["same_fachgruppe_firms_access_15min"] - focal).clip(lower=0)
        other_firms = (all_firms - same_firms).clip(lower=0)
        pop_ring = (spells["pop_access_15min"] - spells["own_cell_pop"]).clip(lower=0)
        same_ring = (same_firms - spells["own_same"]).clip(lower=0)
        other_ring = (other_firms - spells["own_other"]).clip(lower=0)

        spells["log_own_pop"] = np.log1p(spells["own_cell_pop"])
        spells["log_own_same"] = np.log1p(spells["own_same"])
        spells["log_own_other"] = np.log1p(spells["own_other"])
        spells["log_pop_ring_0_15"] = np.log1p(pop_ring)
        spells["log_same_relative_ring_0_15"] = np.log1p(same_ring) - spells["log_pop_ring_0_15"]
        spells["log_other_relative_ring_0_15"] = np.log1p(other_ring) - spells["log_pop_ring_0_15"]
        spells["log_tt_motorway_exit"] = np.log1p(spells["tt_motorway_exit_min"])
        spells["calendar_year"] = spells["year"] - START_YEAR
        print("Removed the focal location and created the Cox-model covariates.")

    except (FileNotFoundError, ValueError, KeyError, np.linalg.LinAlgError) as error:
        survival_available = False
        survival_status = "PRÜFEN"
        survival_messages.append(str(error))
        print(f"Überlebensprüfung PRÜFEN: {error}")
else:
    print("Überlebens-Prüfschritt nicht ausgeführt; siehe separaten Survival-Preflight/Befund.")

### Create the Cox model frame


In [ ]:
if survival_available:
    try:
        survival_terms = [
            "log_own_pop", "log_own_same", "log_own_other", "log_pop_ring_0_15",
            "log_same_relative_ring_0_15", "log_other_relative_ring_0_15",
            "log_tt_motorway_exit", "walk_pt_routes_10min", "pt_ohne_haltestelle", "calendar_year",
        ]
        survival_frame = spells[[
            "standort_id", "grid_id", "Fachgruppe_ID", "sparte", "sparte_name", "year", "period",
            "start", "stop", "event", *survival_terms,
        ]].copy()
        if not np.isfinite(survival_frame[survival_terms].to_numpy()).all(): raise ValueError("Non-finite Cox covariates.")
        print(f"Final survival sample: {survival_frame.shape[0]:,} intervals × {survival_frame.shape[1]} columns")
        print(f"Locations: {survival_frame['standort_id'].nunique():,}; exits: {int(survival_frame['event'].sum()):,}")
        print(f"Cells: {survival_frame['grid_id'].nunique():,}; Fachgruppe strata: {survival_frame['Fachgruppe_ID'].nunique()}")

    except (FileNotFoundError, ValueError, KeyError, np.linalg.LinAlgError) as error:
        survival_available = False
        survival_status = "PRÜFEN"
        survival_messages.append(str(error))
        print(f"Überlebensprüfung PRÜFEN: {error}")
else:
    print("Überlebens-Prüfschritt nicht ausgeführt; siehe separaten Survival-Preflight/Befund.")

### Survival-model checks


In [ ]:
if survival_available:
    try:
        survival_correlations = correlation_report(survival_frame, survival_terms)
        events_by_group = survival_frame.groupby("Fachgruppe_ID")["event"].agg(["sum", "size"])
        survival_summary = pd.Series({
            "intervals": len(survival_frame), "columns": survival_frame.shape[1],
            "locations": survival_frame["standort_id"].nunique(), "events": int(survival_frame["event"].sum()),
            "cells": survival_frame["grid_id"].nunique(), "fachgruppe_strata": len(events_by_group),
            "strata_without_events": int((events_by_group["sum"]==0).sum()),
            "strata_below_30_events": int((events_by_group["sum"]<30).sum()),
            "events_per_covariate": float(survival_frame["event"].sum()/len(survival_terms)),
            "condition_number": condition_number(survival_frame, survival_terms),
        })
        display(survival_summary.to_frame("value"))
        print("Correlations with absolute value at least 0.70:")
        print(survival_correlations.to_string() if len(survival_correlations) else "None")
        survival_summary.to_csv(RESULT_DIR/"survival_input_diagnostics.csv", header=["value"])
        survival_correlations.to_csv(RESULT_DIR/"survival_high_correlations.csv", header=True)
        print("Saved 2 diagnostic files.")

    except (FileNotFoundError, ValueError, KeyError, np.linalg.LinAlgError) as error:
        survival_available = False
        survival_status = "PRÜFEN"
        survival_messages.append(str(error))
        print(f"Überlebensprüfung PRÜFEN: {error}")
else:
    print("Überlebens-Prüfschritt nicht ausgeführt; siehe separaten Survival-Preflight/Befund.")

## Zusammenfassung der NB2-Checkliste

In [ ]:
checklist_summary = pd.DataFrame(checklist)
checklist_summary["Reihenfolge"] = checklist_summary["Prüfpunkt"].map(lambda s: tuple(map(int, s.split("."))))
checklist_summary = checklist_summary.sort_values("Reihenfolge").drop(columns="Reihenfolge").reset_index(drop=True)
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(checklist_summary)
print("Statushäufigkeiten:")
print(checklist_summary.Status.value_counts().to_string())
unresolved = checklist_summary.loc[checklist_summary.Status.isin(["PRÜFEN", "OFFEN"])]
print(f"{len(unresolved)} auffällige/offene Punkte; keine pauschale Freigabe allein anhand dieser Screens.")
if len(unresolved):
    print("Weiter bearbeiten:", ", ".join(unresolved["Prüfpunkt"]))
print("Hauptmodelle: alle gültigen Modellzeilen; zusätzliche Modelltests: feste Zellstichprobe. "
      "Die Ergebnisse stehen direkt in den Zell-Outputs. Cox: bestehende Eingangsprüfungen oben; "
      "Proportional-Hazards-Diagnostik weiterhin in Notebook 09.")

# Vergleichbare Herkunfts- und Stichprobenangaben direkt im gespeicherten Notebook.
retained_periods = sorted(founding.period.astype(str).unique().tolist())
run_metadata = {
    "resolution_m": CELL_SIZE, "input_profile": INPUT_PROFILE,
    "data_directory": str(DATA_DIR), "feature_directory": str(FEATURE_DIR),
    "analysis_years": [START_YEAR, END_YEAR], "lead_in_year": LAG_YEAR,
    "raw_rows": int(len(raw_panel)), "raw_cells": int(raw_panel.grid_id.nunique()),
    "raw_births": int(raw_panel.births.sum()),
    "model_rows": int(len(founding)), "model_cells": int(founding.grid_id.nunique()),
    "model_births": int(y_founding.sum()),
    "excluded_rows": int(len(raw_panel)-len(founding)),
    "excluded_births": int(raw_panel.births.sum()-y_founding.sum()),
    "retained_periods": retained_periods,
    "missing_periods": sorted(set(expected_periods)-set(retained_periods)),
    "model_terms": list(X_check.columns), "removed_constant_columns": dropped_constants,
    "offset": "none; absolute counts", "standard_errors": "grid-cell clustered",
    "time_effects": "full quarter fixed effects", "random_seed": RANDOM_SEED,
    "robustness_max_cells": ROBUSTNESS_CELLS, "moran_permutations": MORAN_PERMUTATIONS,
    "compatibility_mappings": compatibility_mappings,
    "survival_status": survival_status, "survival_messages": survival_messages,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "software": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__},
}
display(pd.Series({k: v for k, v in run_metadata.items() if k not in ["compatibility_mappings", "software"]}, dtype=object).to_frame("Lauf / Stichprobe"))
print("Angewandte Kompatibilitätszuordnungen:")
display(pd.DataFrame(compatibility_mappings) if compatibility_mappings else pd.DataFrame({"Hinweis": ["Keine Zuordnung erforderlich."]}))
print(f"Überlebensprüfungen: {survival_status}")
for message in survival_messages: print(message)
print("Vergleich zwischen Auflösungen: Nullanteil, Dispersion, AIC und rohe Zählfehler hängen von der Aggregation ab. "
      "AIC vergleicht hier Poisson/NB2 innerhalb derselben Auflösung und Stichprobe; kein AIC-Ranking der Auflösungen. "
      "Abweichende Abdeckung, Ausschlüsse oder Datenaufbereitung separat berücksichtigen.")
checklist_summary.to_csv(RESULT_DIR / "nb2_checklist_summary.csv", index=False)
(RESULT_DIR / "assumption_run_metadata.json").write_text(json.dumps(run_metadata, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Diagnoseexporte: {RESULT_DIR}")
print("Nach Run All dieses Notebook speichern (Strg+S). Die gespeicherten Zell-Outputs bleiben beim Wiederöffnen sichtbar; "
      "zum Lesen ist keine Neuschätzung nötig. Diese Sitzung schätzt nur Modelle auf bestehenden Daten, kein neues Routing.")